[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fvalenzuelag/Ingenier-a-de-Soluciones-con-Inteligencia-Artificial/blob/main/RA2/IL2.3/1-basic-planning-agent.ipynb)


# 1. Planificación Básica de Agentes

## Objetivos de Aprendizaje
- Comprender qué es la planificación en el contexto de los agentes de IA.
- Diferenciar entre un agente reactivo (como los vistos hasta ahora) y un agente planificador.
- Implementar un agente básico de tipo **Plan-and-Execute**.
- Entender las limitaciones de la planificación estática y la necesidad de la orquestación.

## De la Reacción a la Planificación

En los módulos anteriores, hemos construido agentes que son fundamentalmente **reactivos**. El ciclo ReAct (Reason + Act) es potente, pero solo decide la siguiente mejor acción a tomar. Es como un trabajador que, después de cada tarea, vuelve a preguntar: '¿Y ahora qué hago?'

Esto funciona para tareas simples, pero falla ante objetivos complejos que requieren múltiples pasos interdependientes. Por ejemplo, si le pedimos a un agente: 'Investiga sobre la empresa NVIDIA, encuentra el nombre de su CEO y luego busca la biografía de esa persona en Wikipedia', un agente puramente reactivo podría perderse.

La **planificación** es la capacidad de un agente para descomponer un objetivo complejo en una secuencia de pasos más pequeños y ejecutables **antes** de empezar a actuar. Es la diferencia entre reaccionar al momento y tener una estrategia.

### Tipos de Agentes Planificadores

Existen varias arquitecturas para la planificación, pero dos de las más comunes son:

1.  **Agentes Plan-and-Execute**:
    - **Cómo funcionan**: Primero, llaman a un LLM para crear un plan completo, paso a paso. Luego, ejecutan esos pasos en orden, sin desviarse del plan original.
    - **Ventajas**: Simple de implementar y predecible.
    - **Desventajas**: El plan es estático. Si un paso falla o la información obtenida en un paso intermedio invalida el resto del plan, el agente no puede adaptarse.

2.  **Agentes ReAct (y similares)**:
    - **Cómo funcionan**: Aunque los hemos llamado reactivos, también son una forma de planificación incremental. Planean y ejecutan un paso a la vez, y el resultado de cada acción influye en la planificación del siguiente paso.
    - **Ventajas**: Son dinámicos y pueden corregir el rumbo sobre la marcha.
    - **Desventajas**: Pueden entrar en bucles o perder de vista el objetivo general si se desvían demasiado.

En este notebook, construiremos un agente **Plan-and-Execute** básico para ilustrar el concepto de forma clara.

### 1. Instalación y Configuración

In [3]:
# --- Instalación de dependencias (se ejecuta solo en Google Colab) ---
# En local no hace nada: usa `pip install -r requirements.txt` desde la raíz del repo.
import sys
if "google.colab" in sys.modules:
    !pip install -q langchain-openai python-dotenv wikipedia


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.4/146.4 kB 7.1 MB/s eta 0:00:00


In [4]:
# --- Credenciales: funciona en local (.env) y en Google Colab (Secrets) ---
import os
try:
    from google.colab import userdata          # Colab: panel 🔑 Secrets
    # Solo LLM_API_KEY es obligatorio. Los demás son opcionales: defínelos como
    # Secrets únicamente si quieres usar otro proveedor o modelo.
    for _k in ("LLM_API_KEY", "GOOGLE_API_KEY", "LANGSMITH_API_KEY",
               "LLM_BASE_URL", "LLM_MODEL", "LLM_MODEL_SMALL"):
        try:
            os.environ[_k] = userdata.get(_k)
        except Exception:
            pass                                # el Secret no existe: se usa el default
    os.environ.setdefault("LLM_BASE_URL", "https://api.groq.com/openai/v1")
    os.environ.setdefault("LLM_MODEL", "openai/gpt-oss-120b")
    os.environ.setdefault("LLM_MODEL_SMALL", "openai/gpt-oss-120b")
except ImportError:
    from dotenv import load_dotenv             # Local: archivo .env en la raíz
    load_dotenv()

import os
import wikipedia
from langchain_openai import ChatOpenAI

# Configurar el idioma de Wikipedia
wikipedia.set_lang("es")

# Configuración del LLM
try:
    llm = ChatOpenAI(
        model=os.getenv("LLM_MODEL", "openai/gpt-oss-120b"),
        openai_api_base=os.environ.get("LLM_BASE_URL"),
        openai_api_key=os.environ.get("LLM_API_KEY"),
        temperature=0
    )
    print("✅ LLM de LangChain configurado.")
except Exception as e:
    print(f"❌ Error configurando el LLM: {e}")
    llm = None

✅ LLM de LangChain configurado.


### 2. El Planificador: Creando el Plan

Nuestro primer paso es usar el LLM no para ejecutar una tarea, sino para **crear un plan**. Le daremos un objetivo y le pediremos que nos devuelva una lista de pasos claros y concisos.

In [5]:
from langchain_core.prompts import ChatPromptTemplate

planner_prompt = ChatPromptTemplate.from_messages([
    ("system", "Eres un excelente planificador de tareas. Tu trabajo es tomar un objetivo complejo y descomponerlo en una lista de pasos simples y ejecutables. Responde únicamente con la lista de pasos."),
    ("human", "Objetivo: {goal}")
])

planner_chain = planner_prompt | llm

goal = "Investiga sobre la empresa Tesla, encuentra el nombre de su CEO y luego busca la biografía de esa persona en Wikipedia."

plan = planner_chain.invoke({"goal": goal})

print("--- PLAN GENERADO ---")
print(plan.content)

--- PLAN GENERADO ---
1. Abre tu navegador web preferido.  
2. Busca “Tesla” en un motor de búsqueda (por ejemplo, Google).  
3. Accede a la página oficial de Tesla (https://www.tesla.com) o a la sección “Acerca de” para obtener información actualizada sobre la empresa.  
4. Localiza la información del equipo ejecutivo; anota el nombre del CEO (actualmente Elon Musk).  
5. Copia el nombre completo del CEO.  
6. En el motor de búsqueda, escribe “Elon Musk Wikipedia”.  
7. Haz clic en el enlace que lleva a la página de Wikipedia de Elon Musk.  
8. Lee y recopila la biografía del CEO desde la sección introductoria y los apartados relevantes (infancia, educación, carrera, etc.).  
9. Guarda o copia la información biográfica que necesites para tu investigación.


### 3. El Ejecutor: Siguiendo el Plan

Ahora que tenemos un plan en formato de texto, necesitamos un **ejecutor**. El ejecutor leerá cada paso del plan y utilizará una herramienta (en este caso, una simple búsqueda en Wikipedia) para completarlo.

Para este ejemplo básico, nuestro ejecutor será una simple función de Python que interpreta cada línea del plan.

In [6]:
def execute_plan(plan_text):
    steps = plan_text.strip().split('\n')
    results = []

    print("--- EJECUTANDO PLAN ---")
    for i, step in enumerate(steps):
        # Extraer la acción y el sujeto de cada paso (esto es muy básico, en un agente real se usarían herramientas más robustas)
        clean_step = step.split('.', 1)[-1].strip() # Quita el número del paso
        print(f"[Paso {i+1}] Ejecutando: {clean_step}")

        try:
            # Asumimos que cada paso es una búsqueda en Wikipedia
            # En un agente real, aquí se seleccionaría la herramienta adecuada
            query = clean_step.split('sobre ')[-1].split(' de ')[-1] # Intenta adivinar el sujeto de la búsqueda
            result = wikipedia.summary(query, sentences=2)
            print(f"Resultado: {result}")
            results.append(result)
        except Exception as e:
            error_message = f"No se pudo completar el paso: {e}"
            print(error_message)
            results.append(error_message)

    return results

# Ejecutamos el plan generado por el LLM
execution_results = execute_plan(plan.content)

--- EJECUTANDO PLAN ---
[Paso 1] Ejecutando: Abre tu navegador web preferido.
Resultado: Internet Explorer​ (anteriormente Microsoft Internet Explorer​ y Windows Internet Explorer,​ usualmente abreviado a IE) es una serie descontinuada de navegadores web gráficos desarrollado por Microsoft para el sistema operativo Microsoft Windows desde 1995. Se lanzó por primera vez como parte del paquete complementario "Plus!" para Windows 95 ese año.
[Paso 2] Ejecutando: Busca “Tesla” en un motor de búsqueda (por ejemplo, Google).
No se pudo completar el paso: Expecting value: line 1 column 1 (char 0)
[Paso 3] Ejecutando: Accede a la página oficial de Tesla (https://www.tesla.com) o a la sección “Acerca de” para obtener información actualizada sobre la empresa.
No se pudo completar el paso: Expecting value: line 1 column 1 (char 0)
[Paso 4] Ejecutando: Localiza la información del equipo ejecutivo; anota el nombre del CEO (actualmente Elon Musk).
No se pudo completar el paso: Expecting value: line 

In [7]:
import wikipedia
import wikipedia.wikipedia as wikipedia_backend

wikipedia.set_lang("es")


def execute_plan(plan):
    results = []

    print("--- EJECUTANDO PLAN ---")

    for i, step in enumerate(plan, start=1):
        action = step.get("accion")
        query = step.get("consulta", "").strip()

        print(f"\n[Paso {i}] Acción: {action}")
        print(f"Consulta: {query}")

        # El ejecutor solo admite consultas a Wikipedia.
        if action != "buscar_wikipedia" or not query:
            print("Paso inválido: revisa la acción y la consulta.")
            continue

        try:
            result = wikipedia.summary(
                query,
                sentences=3,
                auto_suggest=False
            )

            print(f"Resultado: {result}")
            results.append(result)

        except wikipedia.exceptions.DisambiguationError as e:
            print(f"Consulta ambigua. Opciones: {e.options[:5]}")

        except wikipedia.exceptions.PageError:
            print(f"No se encontró el artículo: {query}")

        except Exception as e:
            print(f"Error al consultar Wikipedia: {type(e).__name__}: {e}")

    return results


# Ejemplo para comprobar el ejecutor.
# Estas consultas son explícitas; aquí no se identifica al CEO actual.
plan = [
    {"accion": "buscar_wikipedia", "consulta": "Tesla, Inc."},
    {"accion": "buscar_wikipedia", "consulta": "Elon Musk"}
]

execution_results = execute_plan(plan)

--- EJECUTANDO PLAN ---

[Paso 1] Acción: buscar_wikipedia
Consulta: Tesla, Inc.
Error al consultar Wikipedia: JSONDecodeError: Expecting value: line 1 column 1 (char 0)

[Paso 2] Acción: buscar_wikipedia
Consulta: Elon Musk
Error al consultar Wikipedia: JSONDecodeError: Expecting value: line 1 column 1 (char 0)


### 4. Conclusiones y Transición a la Orquestación

Hemos creado con éxito un agente que puede planificar y luego ejecutar. Este es un gran paso adelante con respecto a un agente puramente reactivo.

Sin embargo, las **limitaciones** de nuestro enfoque simple son evidentes:

1.  **Plan Estático**: El plan no puede cambiar. Si el CEO de Tesla no fuera Elon Musk, el plan seguiría buscando la biografía de Elon Musk.
2.  **Ejecutor Frágil**: Nuestro ejecutor es muy simple. Asume que cada paso es una búsqueda en Wikipedia y extrae el tema de búsqueda de forma muy rudimentaria.
3.  **Falta de Contexto entre Pasos**: El resultado del paso 1 (investigar sobre Tesla) no se utiliza en el paso 2. El agente no "sabe" que el CEO que encontró en el primer paso es la persona que debe buscar en el segundo.

Estos problemas son precisamente los que resuelve la **orquestación de agentes**. Un orquestador es un sistema más inteligente que no solo crea un plan, sino que también:
- Gestiona el flujo de datos entre los pasos.
- Selecciona la herramienta correcta para cada tarea.
- Puede manejar errores y adaptar el plan si es necesario.
- Coordina la colaboración entre diferentes agentes especializados.

En los próximos notebooks, exploraremos cómo frameworks como **LangChain y CrewAI** nos proporcionan herramientas de orquestación para construir agentes mucho más potentes y robustos, superando las limitaciones de nuestro planificador básico.